# Flake Detector Inspection Notebook

Use this notebook to inspect generated samples, masks, and prediction previews.

In [1]:
from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [2]:
root = Path('.')
train_manifest_path = root / 'data' / 'splits' / 'train_manifest.json'
val_manifest_path = root / 'data' / 'splits' / 'val_manifest.json'

if train_manifest_path.exists():
    train_manifest = json.loads(train_manifest_path.read_text())
    print('Train images:', len(train_manifest))
    print('Train instances:', sum(len(x.get('instances', [])) for x in train_manifest))
else:
    print('Run build_dataset.py first to create manifests.')

Train images: 400
Train instances: 1022


In [11]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import cv2
import numpy as np

if train_manifest_path.exists() and train_manifest:
    sample_idx_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(train_manifest) - 1,
        step=1,
        description="Index:",
        continuous_update=False,
    )

    def render(idx):
        sample = train_manifest[idx]
        image = cv2.imread(sample["image_path"], cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        overlay = image.copy()
        for inst in sample.get("instances", []):
            mask = cv2.imread(inst["mask_path"], cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            mask = (mask > 0).astype(np.uint8)
            color = np.array([220, 80, 60], dtype=np.uint8)
            overlay = np.where(
                mask[:, :, None] > 0,
                (0.6 * overlay + 0.4 * color).astype(np.uint8),
                overlay,
            )

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title("Image")
        plt.imshow(image)
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.title("Overlay")
        plt.imshow(overlay)
        plt.axis("off")
        plt.show()

    out = widgets.interactive_output(render, {"idx": sample_idx_slider})
    display(sample_idx_slider, out)

IntSlider(value=0, continuous_update=False, description='Index:', max=399)

Output()